In [98]:
import pandas as pd
import time
import imdb
import os
import requests
import re
from dotenv import load_dotenv
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [99]:
data = {
    'script_text': [
        "EXT. SPACE - DAY. An explosion rocks the galaxy.",
        "INT. CAFE - NIGHT. They talk about love and coffee.",
        "EXT. MOUNTAIN - DAY. The hero climbs the jagged rocks.",
        "INT. OFFICE - DAY. A boring meeting about spreadsheets.",
        "EXT. BATTLEFIELD - NIGHT. Soldiers charge through the mud."
    ],
    'is_successful': [1, 1, 1, 0, 0]
}

df = pd.DataFrame(data)

In [100]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df['script_text'])
bow_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
print("Vocabulary size:", len(vectorizer.get_feature_names_out()))
display(bow_df.head())

Vocabulary size: 24


,battlefield,boring,cafe,charge,climbs,coffee,day,explosion,ext,galaxy,...,meeting,mountain,mud,night,office,rocks,soldiers,space,spreadsheets,talk
0,0,0,0,0,0,0,1,1,1,1,...,0,0,0,0,0,1,0,1,0,0
1,0,0,1,0,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1
2,0,0,0,0,1,0,1,0,1,0,...,0,1,0,0,0,1,0,0,0,0
3,0,1,0,0,0,0,1,0,0,0,...,1,0,0,0,1,0,0,0,1,0
4,1,0,0,1,0,0,0,0,1,0,...,0,0,1,1,0,0,1,0,0,0


In [101]:
y = df['is_successful']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67)

baseline_model = LogisticRegression()
baseline_model.fit(X_train, y_train)

word_weights = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'weight': baseline_model.coef_[0]
}).sort_values(by='weight', ascending=False)

print("Words most associated with success:")
print(word_weights.head(3))

Words most associated with success:
      word    weight
2     cafe  0.264874
5   coffee  0.264874
23    talk  0.264874


In [102]:
movie_titles = [
    "10 Things I Hate About You", "12", "12 and Holding", "12 Monkeys", "12 Years a Slave",
    "127 Hours", "1492: Conquest of Paradise", "15 Minutes", "17 Again", "187",
    "2001: A Space Odyssey", "2012", "25th Hour", "30 Minutes or Less", "42",
    "44 Inch Chest", "48 Hrs.", "50/50", "500 Days of Summer", "8mm", "9",
    "A Few Good Men", "A Serious Man", "Above the Law", "Absolute Power", "The Abyss",
    "Ace Ventura: Pet Detective", "Adaptation", "The Addams Family", "The Adjustment Bureau",
    "The Adventures of Buckaroo Banzai Across the Eighth Dimension", "Affliction",
    "After School Special", "After.Life", "Agnes of God", "Air Force One", "Airplane",
    "Airplane 2: The Sequel", "Aladdin", "Ali", "Alien 3", "Alien Nation", "Alien vs. Predator",
    "Alien: Resurrection", "Aliens", "All About Eve", "All About Steve", "All the King's Men",
    "All the President's Men", "Almost Famous", "Alone in the Dark", "Amadeus", "Amelia",
    "American Beauty", "American Gangster", "American Graffiti", "American History X",
    "American Hustle", "American Madness", "American Outlaws", "American Pie",
    "The American President", "American Psycho", "American Shaolin: King of Kickboxers II",
    "American Splendor", "An American Werewolf in London", "The American", "The Amityville Asylum",
    "Amour", "An Education", "Analyze That", "Analyze This", "Anastasia", "Angel Eyes",
    "Angels & Demons", "Anna Karenina", "Annie Hall", "The Anniversary Party", "Anonymous",
    "Antitrust", "Antz", "The Apartment", "Apocalypse Now", "April Fool's Day", "Apt Pupil",
    "Arbitrage", "Arcade", "Arctic Blue", "Argo", "Armageddon", "Army of Darkness",
    "Arsenic and Old Lace", "Arthur", "The Artist", "As Good As It Gets", "Assassins",
    "The Assignment", "At First Sight", "August: Osage County", "Austin Powers - International Man of Mystery",
    "Austin Powers - The Spy Who Shagged Me", "Authors Anonymous", "Autumn in New York", "Avatar",
    "The Avengers", "The Avengers", "L'Avventura", "Awakenings",
    "Babel", "Bachelor Party", "The Bachelor Party", "The Back-up Plan", "Backdraft", "Bad Boys",
    "Bad Country", "Bad Day at Black Rock", "Bad Dreams", "Bad Lieutenant", "Bad Santa",
    "Bad Teacher", "Badlands", "Bamboozled", "Barry Lyndon", "Barton Fink", "Basic",
    "Basic Instinct", "Basquiat", "Batman", "Batman Returns", "The Battle of Algiers",
    "The Battle of Shaker Heights", "Battle: Los Angeles", "The Beach", "Bean",
    "Beasts of the Southern Wild", "Beavis and Butt-head Do America", "Beginners", "Being Human",
    "Being John Malkovich", "Being There", "The Believer", "Beloved", "Benny & Joon",
    "The Best Exotic Marigold Hotel", "Big", "The Big Blue", "Big Fish", "The Big Lebowski",
    "The Big White", "The Birds", "Birthday Girl", "The Black Dahlia", "Black Rain",
    "Black Snake Moan", "Black Swan", "Blade", "Blade II", "Blade Runner", "Blade: Trinity",
    "The Blast from the Past", "The Blind Side", "The Bling Ring", "Blood and Wine",
    "Blood Simple", "Blow", "Blue Valentine", "Blue Velvet", "Bodies, Rest & Motion", "Body Heat",
    "Body of Evidence", "The Bodyguard", "Bones", "The Bonfire of the Vanities", "Bonnie and Clyde",
    "Boogie Nights", "The Book of Eli", "Boondock Saints 2: All Saints Day", "The Boondock Saints",
    "Bottle Rocket", "Bound", "The Bounty Hunter", "The Bourne Identity", "The Bourne Supremacy",
    "The Bourne Ultimatum", "The Box", "Braveheart", "Brazil", "Break", "Breakdown",
    "The Breakfast Club", "Breaking Away", "Brick", "Bridesmaids", "Bringing Out the Dead",
    "Broadcast News", "Broken Arrow", "Broken Embraces", "The Brothers Bloom", "Bruce Almighty",
    "Buffy the Vampire Slayer", "Bull Durham", "Buried", "Burlesque", "Burn After Reading",
    "Burning Annie", "The Butterfly Effect", "The Cable Guy", "Candle to Water", "Capote", "Carrie",
    "Cars 2", "Case 39", "Casino", "Cast Away", "Catch Me If You Can", "Catwoman", "Cecil B. Demented",
    "Cedar Rapids", "Celeste & Jesse Forever", "The Cell", "Cellular", "The Change-Up",
    "Changeling", "Chaos", "Charade", "Charlie's Angels", "Chasing Amy", "Chasing Sleep",
    "Cherry Falls", "Chinatown", "Christ Complex", "Chronicle",
    "The Chronicles of Narnia: The Lion, the Witch and the Wardrobe", "The Cider House Rules",
    "The Cincinnati Kid", "Cinema Paradiso", "Cirque du Freak: The Vampire's Assistant",
    "Citizen Kane", "City of Joy", "Clash of the Titans", "Clerks", "Cliffhanger", "Clueless", "Cobb",
    "Code of Silence", "Cold Mountain", "Collateral", "Collateral Damage", "Colombiana",
    "Color of Night", "Commando", "Conan the Barbarian", "Confessions of a Dangerous Mind",
    "Confidence", "Constantine", "The Cooler", "Copycat", "Coraline", "Coriolanus",
    "Cowboys & Aliens", "Cradle 2 the Grave", "Crank", "Crash", "Crazy, Stupid, Love", "Crazylove",
    "Creation", "Crime Spree", "The Croods", "Crouching Tiger, Hidden Dragon", "Croupier",
    "The Crow Salvation", "The Crow", "The Crow: City of Angels", "Cruel Intentions", "The Crying Game",
    "Cube", "The Curious Case of Benjamin Button", "Custody", "The Damned United",
    "Dances with Wolves", "Dark City", "The Dark Knight Rises", "Dark Star", "Darkman", "Date Night",
    "Dave Barry's Complete Guide to Guys", "Dawn of the Dead", "Day of the Dead",
    "The Day the Clown Cried", "The Day the Earth Stood Still", "Days of Heaven",
    "Dead Poets Society", "Death at a Funeral", "Death to Smoochy", "The Debt", "Deception",
    "Deep Cover", "Deep Rising", "The Deer Hunter", "Defiance", "The Departed", "The Descendants",
    "Despicable Me 2", "Detroit Rock City", "Devil in a Blue Dress", "The Devil's Advocate", "Die Hard",
    "Die Hard 2", "Diner", "The Distinguished Gentleman", "Disturbia", "Django Unchained",
    "Do The Right Thing", "Dog Day Afternoon", "Dogma", "Donnie Brasco", "The Doors",
    "Double Indemnity", "Drag Me to Hell", "Dragonslayer", "Drive", "Drive Angry",
    "Drop Dead Gorgeous", "A Dry White Season", "Duck Soup", "Dumb and Dumber", "Dune", "E.T. the Extra-Terrestrial",
    "Eagle Eye", "Eastern Promises", "Easy A", "Ed TV", "Ed Wood", "Edward Scissorhands",
    "Eight Legged Freaks", "El Mariachi", "Election", "The Elephant Man", "Elizabeth: The Golden Age",
    "Enemy of the State", "The English Patient", "Enough", "Entrapment", "Erik the Viking",
    "Erin Brockovich", "Escape From L.A.", "Escape From New York",
    "Eternal Sunshine of the Spotless Mind", "Even Cowgirls Get the Blues", "Event Horizon",
    "Evil Dead", "Evil Dead II", "Excalibur", "eXistenZ", "Extract",
    "The Fabulous Baker Boys", "Face/Off", "Fair Game", "The Family Man", "Fantastic Four",
    "Fantastic Mr. Fox", "Fargo", "Fast Times at Ridgemont High", "Fatal Instinct",
    "The Fault in Our Stars", "Fear and Loathing in Las Vegas", "Feast", "Ferris Bueller's Day Off",
    "Field of Dreams", "The Fifth Element", "Fight Club", "The Fighter", "Final Destination",
    "Final Destination 2", "Finding Nemo", "Five Easy Pieces", "Flash Gordon", "Fletch", "Flight",
    "The Flintstones", "Forrest Gump", "The Four Feathers", "Four Rooms", "Foxcatcher", "Fracture",
    "Frances", "Frankenstein", "Freaked", "Freddy vs. Jason", "The French Connection", "Frequency",
    "Friday the 13th", "Friday the 13th Part VIII: Jason Takes Manhattan", "Fright Night",
    "Fright Night", "From Dusk Till Dawn", "From Here to Eternity", "Frozen",
    "Frozen", "Frozen River", "Fruitvale Station", "The Fugitive", "Funny People",
    "G.I. Jane", "G.I. Joe: The Rise of Cobra", "Game 6", "The Game", "Gamer", "Gandhi",
    "Gang Related", "Gangs of New York", "Garden State", "Gattaca", "Get Carter", "Get Low",
    "Get Shorty", "The Getaway", "Ghost", "The Ghost and the Darkness", "Ghost Rider",
    "Ghost Ship", "Ghost World", "Ghostbusters", "Ghostbusters II", "Ginger Snaps",
    "The Girl with the Dragon Tattoo", "Gladiator", "Glengarry Glen Ross", "Go", "The Godfather",
    "The Godfather Part II", "The Godfather Part III", "Gods and Monsters", "Godzilla",
    "Gone in 60 Seconds", "The Good Girl", "Good Will Hunting", "Gothika", "The Graduate",
    "Gran Torino", "Grand Hotel", "Grand Theft Parsons", "The Grapes of Wrath", "Gravity",
    "The Green Mile", "Gremlins", "Gremlins 2: The New Batch", "The Grifters", "Grosse Pointe Blank",
    "Groundhog Day", "The Grudge", "Hackers", "Hall Pass", "Halloween: The Curse of Michael Myers",
    "Hancock", "The Hangover", "Hanna", "Hannah and Her Sisters", "Hannibal",
    "Happy Birthday, Wanda June", "Happy Feet", "Hard Rain", "Hard to Kill",
    "Harold & Kumar Go to White Castle", "The Haunting", "He's Just Not That Into You", "Heat",
    "Heathers", "Heavenly Creatures", "Heavy Metal", "The Hebrew Hammer", "Heist",
    "Hellbound: Hellraiser II", "Hellboy", "Hellboy II: The Golden Army", "Hellraiser",
    "Hellraiser III: Hell on Earth", "Hellraiser: Deader", "Hellraiser: Hellseeker", "The Help",
    "Henry Fool", "Henry's Crime", "Hesher", "High Fidelity", "Highlander", "Highlander: Endgame",
    "The Hills Have Eyes", "His Girl Friday", "Hitchcock", "The Hitchhiker's Guide to the Galaxy",
    "Hollow Man", "Honeydripper", "Horrible Bosses", "The Horse Whisperer", "The Hospital",
    "Hostage", "Hot Tub Time Machine", "Hotel Rwanda", "House of 1000 Corpses",
    "How to Lose Friends & Alienate People", "How to Train Your Dragon", "How to Train Your Dragon 2",
    "Hudson Hawk", "The Hudsucker Proxy", "Human Nature", "The Hunt for Red October",
    "I Am Number Four", "I Am Sam", "I Love You Phillip Morris", "I Spit on Your Grave",
    "I Still Know What You Did Last Summer", "I'll Do Anything", "I, Robot", "The Ice Storm",
    "The Ides of March", "The Imaginarium of Doctor Parnassus", "In the Bedroom", "In the Loop",
    "Inception", "The Incredibles", "Independence Day", "Indiana Jones and the Last Crusade",
    "Raiders of the Lost Ark", "Indiana Jones and the Temple of Doom",
    "Indiana Jones and the Kingdom of the Crystal Skull", "The Informant!", "Inglourious Basterds", "The Insider", "Insidious",
    "Insomnia", "Interstellar", "Interview with the Vampire", "Into the Wild", "Intolerable Cruelty",
    "Inventing the Abbotts", "The Invention of Lying", "Invictus", "The Iron Lady", "The Island",
    "It Happened One Night", "It's a Wonderful Life", "It's Complicated", "The Italian Job",
    "The Jacket", "Jackie Brown", "Jacob's Ladder", "Jane Eyre", "Jason X", "Jaws", "Jaws 2",
    "Jay and Silent Bob Strike Back", "Jennifer 8", "Jennifer's Body", "Jerry Maguire", "JFK",
    "Jimmy and Judy", "John Q", "Judge Dredd", "Juno", "Jurassic Park", "Jurassic Park III",
    "The Lost World: Jurassic Park", "Kafka", "Kalifornia", "Kate & Leopold", "Kids",
    "The Kids Are All Right", "Kill Bill", "Killing Zoe", "King Kong",
    "The King of Comedy", "The King's Speech", "The Kingdom", "Klute", "Knocked Up",
    "Kramer vs. Kramer", "Kundun", "Kung Fu Panda", "L.A. Confidential", "Labor of Love", "Labyrinth",
    "The Ladykillers", "Lake Placid", "Land of the Dead", "Larry Crowne", "The Last Boy Scout",
    "Last Chance Harvey", "The Last Flight", "The Last of the Mohicans", "The Last Samurai",
    "The Last Station", "Last Tango in Paris", "Law Abiding Citizen", "Leaving Las Vegas",
    "Legally Blonde", "Legend", "Legion", "Les Misérables", "Leviathan", "Liar Liar", "Life",
    "Life as a House", "The Life of David Gale", "Life of Pi", "Light Sleeper", "The Limey",
    "Limitless", "Lincoln", "The Lincoln Lawyer", "Little Athens", "The Little Mermaid",
    "Little Nicky", "Living in Oblivion", "Lock, Stock and Two Smoking Barrels", "Logan's Run",
    "Lone Star", "The Long Kiss Goodnight", "Looper", "Lord of Illusions",
    "The Lord of the Rings: The Fellowship of the Ring", "The Lord of the Rings: The Return of the King",
    "The Lord of the Rings: The Two Towers", "Lord of War", "The Losers", "Lost Highway",
    "Lost Horizon", "Lost in Space", "Lost in Translation", "Love & Basketball", "Machete",
    "Machine Gun Preacher", "Mad Max 2", "Made", "Magnolia",
    "The Majestic", "Major League", "Malcolm X", "Malibu's Most Wanted",
    "The Man in the Iron Mask", "Man on Fire", "Man on the Moon", "Man Trouble",
    "The Man Who Knew Too Much", "The Man Who Wasn't There", "The Manchurian Candidate",
    "Manhattan Murder Mystery", "Manhunter", "Margaret", "Margin Call", "Margot at the Wedding",
    "El Mariachi", "Marley & Me", "Martha Marcy May Marlene", "Marty", "Mary Poppins", "The Mask",
    "Master and Commander: The Far Side of the World", "The Master", "The Matrix Reloaded", "The Matrix", "Max Payne",
    "Mean Streets", "The Mechanic", "Meet Joe Black", "Meet John Doe", "Megamind", "Memento",
    "Men in Black", "Men in Black 3", "The Men Who Stare at Goats", "Metro", "Miami Vice",
    "Midnight Cowboy", "Midnight Express", "Midnight in Paris",
    "Mighty Morphin Power Rangers: The Movie", "Milk", "Miller's Crossing", "Mimic",
    "Mini's First Time", "Minority Report", "The Miracle Worker", "Mirrors", "Misery",
    "Mission: Impossible", "Mission: Impossible II", "Mission to Mars", "Moneyball", "Monkeybone",
    "Monte Carlo", "Moon", "Moonrise Kingdom", "Moonstruck", "Mr. Blandings Builds His Dream House",
    "Mr. Brooks", "Mr. Deeds Goes to Town", "Mrs. Brown", "Mud", "Mulan", "Mulholland Drive",
    "Mumford", "The Mummy", "Music of the Heart", "Mute Witness", "My Best Friend's Wedding",
    "My Girl", "My Mother Dreams the Satan's Disciples in New York", "My Week with Marilyn",
    "Mystery Men", "Nashville", "Natural Born Killers", "Never Been Kissed", "The NeverEnding Story",
    "New York Minute", "Newsies", "Next", "Next Friday", "The Next Three Days", "Nick of Time",
    "Night Time", "Nightbreed", "The Nightmare Before Christmas",
    "A Nightmare on Elm Street", "A Nightmare on Elm Street: The Dream Child", "Nine", "The Nines",
    "Ninja Assassin", "Ninotchka", "The Ninth Gate", "No Strings Attached", "Notting Hill",
    "Nurse Betty", "O Brother, Where Art Thou?", "Oblivion", "Observe and Report", "Obsessed",
    "Ocean's Eleven", "Ocean's Twelve", "Office Space", "The Omega Man", "One Flew Over the Cuckoo's Nest",
    "Only God Forgives", "Ordinary People", "Orgy of the Dead", "Orphan", "The Other Boleyn Girl",
    "Out of Sight", "The Pacifier", "Pandorum", "Panic Room", "Papadopoulos & Sons", "ParaNorman",
    "Pariah", "The Passion of Joan of Arc", "The Patriot", "Paul", "Pearl Harbor", "Peeping Tom",
    "Peggy Sue Got Married", "Perfect Creature", "A Perfect World", "The Perks of Being a Wallflower",
    "Pet Sematary", "Pet Sematary II", "Petulia", "Philadelphia", "Phone Booth", "Pi", "The Pianist",
    "The Piano", "Pineapple Express", "Pirates of the Caribbean: The Curse of the Black Pearl",
    "Pirates of the Caribbean: Dead Man's Chest", "Pitch Black", "Planet of the Apes",
    "Platinum Blonde", "Platoon", "Pleasantville", "Point Break", "Pokémon: Mewtwo Returns",
    "The Postman", "The Power of One", "Precious", "Predator", "Pretty Woman",
    "Pride & Prejudice", "Priest", "The Princess Bride", "The Private Life of Sherlock Holmes",
    "The Producers", "The Program", "Prom Night", "Prometheus", "The Prophecy", "The Proposal",
    "Psycho", "Public Enemies", "Pulp Fiction", "Punch-Drunk Love", "Purple Rain", "Quantum Project",
    "Queen of the Damned", "The Queen", "Rachel Getting Married", "Raging Bull", "Raising Arizona",
    "Rambling Rose", "Rambo: First Blood Part II", "The Reader", "Real Genius",
    "Rear Window", "Rebel Without a Cause", "Red Planet", "Red Riding Hood", "Reindeer Games",
    "The Relic", "Remember Me", "The Replacements", "Repo Man", "The Rescuers Down Under",
    "Reservoir Dogs", "Resident Evil", "Return of the Apes", "The Revenant", "Revolutionary Road",
    "Ringu", "Rise of the Guardians", "Rise of the Planet of the Apes", "RKO 281", "The Road",
    "Robin Hood: Prince of Thieves", "The Rock", "RocknRolla", "Rocky",
    "The Rocky Horror Picture Show", "Romeo + Juliet", "Ronin", "The Roommate", "Roughshod",
    "The Ruins", "Runaway Bride", "Rush", "Rush Hour", "Rush Hour 2", "Rushmore", "Rust and Bone",
    "S. Darko", "The Saint", "The Salton Sea", "The Sandlot", "Save the Last Dance",
    "Saving Mr. Banks", "Saving Private Ryan", "Saw", "Scarface", "Schindler's List",
    "Scott Pilgrim vs. the World", "Scream", "Scream 2", "Scream 3", "Se7en", "The Searchers",
    "Semi-Pro", "Sense and Sensibility", "Serenity", "Serial Mom", "The Sessions",
    "The Seventh Seal", "Sex and the City", "Sex, Lies, and Videotape", "Sexual Life",
    "Shakespeare in Love", "Shallow Grave", "Shame", "Shampoo", "The Shawshank Redemption",
    "She's Out of My League", "Sherlock Holmes", "Shifty", "The Shining", "The Shipping News",
    "Shivers", "Shrek", "Shrek the Third", "Sideways", "The Siege", "Signs", "The Silence of the Lambs",
    "Silver Bullet", "Silver Linings Playbook", "S1m0ne", "Single White Female", "Sister Act",
    "Six Degrees of Separation", "The Sixth Sense", "Sleepless in Seattle", "Sleepy Hollow",
    "Sling Blade", "Slither", "Slumdog Millionaire", "Smashed", "Smokin' Aces", "Snatch",
    "Snow Falling on Cedars", "Snow White and the Huntsman", "So I Married an Axe Murderer",
    "The Social Network", "Solaris", "Soldier", "Someone to Watch Over Me", "Something's Gotta Give",
    "Source Code", "South Park: Bigger, Longer & Uncut", "Spanglish", "Spare Me", "Spartan", "Speed Racer", "Sphere",
    "Spider-Man", "St. Elmo's Fire", "Star Trek", "Star Trek II: The Wrath of Khan",
    "Star Trek: First Contact", "Star Trek: Generations", "Star Trek: Nemesis",
    "Star Trek: The Motion Picture", "Star Wars: Episode IV - A New Hope", "Star Wars: Episode II - Attack of the Clones",
    "Star Wars: Episode VI - Return of the Jedi", "Star Wars: Episode III - Revenge of the Sith",
    "Star Wars: Episode V - The Empire Strikes Back", "Star Wars: Episode VII - The Force Awakens",
    "Star Wars: Episode I - The Phantom Menace", "Starman", "Starship Troopers", "State and Main",
    "Station West", "Stepmom", "The Sting", "Stir of Echoes", "Storytelling", "Strange Days",
    "Strangers on a Train", "The Stunt Man", "Sugar", "Sugar & Spice", "Sunset Blvd.",
    "Sunshine Cleaning", "Super 8", "Superbad", "Supergirl", "The Surfer King", "Surrogates",
    "Suspect Zero", "Sweeney Todd: The Demon Barber of Fleet Street", "The Sweet Hereafter",
    "Sweet Smell of Success", "Swingers", "Swordfish", "Synecdoche, New York", "Syriana",
    "Take Shelter", "Taking Lives", "The Taking of Pelham One Two Three", "Taking Sides",
    "The Talented Mr. Ripley", "Tall in the Saddle", "Tamara Drewe", "Taxi Driver", "Ted",
    "The Terminator", "Terminator 2: Judgment Day", "Terminator Salvation", "The Rage: Carrie 2",
    "Thelma & Louise", "There's Something About Mary", "They", "The Thing",
    "The Things My Father Never Taught Me", "Thirteen Days", "This Boy's Life", "This Is 40",
    "Thor", "Three Kings", "Three Kings", "Three Men and a Baby",
    "The Three Musketeers", "Thunderbirds", "Thunderheart", "Ticker", "Timber Falls",
    "The Time Machine", "Tin Cup", "Tin Men", "Tinker Tailor Soldier Spy", "Titanic", "TMNT",
    "To Sleep with Anger", "Tombstone", "Tomorrow Never Dies", "Top Gun", "Total Recall",
    "The Tourist", "Toy Story", "Traffic", "Training Day", "Trainspotting",
    "Transformers: The Movie", "Tremors", "Tristan & Isolde", "TRON", "TRON: Legacy",
    "Tropic Thunder", "True Grit", "True Lies", "True Romance", "The Truman Show", "Twilight",
    "The Twilight Saga: New Moon", "Twin Peaks", "Twins", "Two for the Money", "U Turn", "The Ugly Truth",
    "Unbreakable", "Under Fire", "Unknown", "Up", "Up in the Air", "The Usual Suspects",
    "V for Vendetta", "Valkyrie", "Vanilla Sky", "The Verdict", "Very Bad Things", "The Village",
    "Virtuosity", "The Visitor", "Wag the Dog", "A Walk to Remember", "Walking Tall", "Wall Street",
    "Wall Street: Money Never Sleeps", "WALL-E", "Wanted", "War Horse", "War of the Worlds",
    "Warm Springs", "Warrior", "Watchmen", "Water for Elephants", "The Way Back", "We Own the Night",
    "What About Bob?", "What Lies Beneath", "When a Stranger Calls", "While She Was Out",
    "The Whistleblower", "White Christmas", "White Jazz", "The White Ribbon", "White Squall",
    "Whiteout", "Who Framed Roger Rabbit", "Who's Your Daddy?", "Wild at Heart", "The Wild Bunch",
    "Wild Hogs", "Wild Things", "Wild Things: Diamonds in the Rough", "Wild Wild West", "Willow",
    "Win Win", "Wind Chill", "Withnail & I", "Witness", "The Wizard of Oz",
    "The Wolf of Wall Street", "Wonder Boys", "The Woodsman", "The World Is Not Enough",
    "The Wrestler", "The X Files", "X-Men", "X-Men Origins: Wolverine", "xXx",
    "Year One", "Yes Man", "You Can Count on Me", "You've Got Mail", "Youth in Revolt",
    "Zero Dark Thirty", "Zerophilia"
]

In [ ]:
load_dotenv()

API_KEY = os.getenv('OMDB_API_KEY')

def clean_title_v2(title):
    t = str(title).replace('"', '').strip()
    
    t = re.sub(r'\s*\(\d{4}\)', '', t)

    t = re.sub(r'\s*\(.*\)', '', t).strip()
    
    patterns = [
        (r'^(.*),\s+The$', r'The \1'),
        (r'^(.*),\s+A$', r'A \1'),
        (r'^(.*),\s+An$', r'An \1'),
        (r'^(.*),\s+L\'$', r"L'\1"),
        (r'^(.*),\s+El$', r'El \1')
    ]
    
    for pattern, replacement in patterns:
        t = re.sub(pattern, replacement, t)
    
    manual_fixes = {
        "50-50": "50/50",
        "Batman 2": "Batman Returns",
        "Indiana Jones IV": "Indiana Jones and the Kingdom of the Crystal Skull",
        "Seven": "Se7en"
    }
    
    return manual_fixes.get(t, t).strip()

def get_omdb_rating(title):
    if not API_KEY:
        return None
    
    search_term = clean_title_v2(title)
    url = "http://www.omdbapi.com/"
    payload = {'t': search_term, 'apikey': API_KEY}
    
    try:
        response = requests.get(url, params=payload).json()
        if response.get('Response') == 'True':
            return response.get('imdbRating')
    except:
        return None
    return None

results = []

for title in tqdm(movie_titles):
    rating = get_omdb_rating(title)
    
    try:
        numeric_rating = float(rating) if rating and rating != "N/A" else None
    except ValueError:
        numeric_rating = None
        
    results.append({
        'original_title': title.replace('"', ''),
        'rating': numeric_rating
    })

df = pd.DataFrame(results)
df['is_successful'] = df['rating'].apply(lambda x: 1 if pd.notnull(x) and x >= 7.0 else 0)
df = df[df['rating'].notna()]

df.to_csv('movie_ratings_final.csv', index=False, quoting=1)

print(f"\nFinished! Matched {df['rating'].notnull().sum()} out of {len(movie_titles)} movies.")
print(df.head(10))

 12%|█▏        | 128/1093 [00:06<00:45, 21.35it/s]

In [104]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('movie_ratings_final.csv')

def to_filesystem_name(title):
    articles = ("The ", "A ", "An ", "L'", "El ", "La ", "Le ")
    for a in articles:
        if title.startswith(a):
            if a == "L'":
                return f"{title[2:]}, L'"
            return f"{title[len(a):]}, {a.strip()}"
    return title

def load_script_text(clean_title):
    fs_title = to_filesystem_name(clean_title)
    
    file_path = f"./screenplay_dataset/Script_{fs_title}.txt"
    
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return f.read()
    return None

print("Loading script texts...")
print('Script_' + df['original_title'])
df['script_text'] = df['original_title'].apply(load_script_text)

df = df.dropna(subset=['script_text'])

tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2))

X = tfidf.fit_transform(df['script_text'])
y = df['is_successful']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=67)

model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\n--- Model Performance ---")
print(classification_report(y_test, y_pred))

Loading script texts...
Series([], Name: original_title, dtype: object)


ValueError: empty vocabulary; perhaps the documents only contain stop words